# Python 101 - Solutions
## Chapter X

---

**For teaching assistants.** This notebook mirrors the exercises in
`../python101_10.ipynb` one for one. Most solutions end with an `assert`, so
running the whole notebook top to bottom is also a self-test: if it runs clean,
every solution still works.

There is usually more than one right answer - if a student's version passes the
same `assert`, it is correct.

In [ ]:
# Run from the chapter folder, so that `helpers`, `./data/...` and `./pics/...`
# resolve exactly the way they do in the lecture notebooks.
import os
import sys

if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print('working directory:', os.getcwd())

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

party_mapping = {
    ('FIDESZ - Magyar Polgári Szövetség'
     '-Kereszténydemokrata Néppárt'): 'FIDESZ-KDNP',
    'Tisztelet és Szabadság Párt': 'TISZA',
    'Mi Hazánk Mozgalom': 'Mi Hazánk',
    'Demokratikus Koalíció': 'DK',
    'Magyar Kétfarkú Kutya Párt': 'MKKP',
    'Jobbik Magyarországért Mozgalom': 'Jobbik',
    ('A SZOLIDARITÁS PÁRTJA'
     '-Magyar Munkáspárt'): 'Munkáspárt',
}

data = pd.read_csv('./data/vote2026.csv')
data['party'] = data['party'].replace(party_mapping)
data.head()

Sanity check on the data before we start - always worth doing in front of them.

In [ ]:
print(f'{len(data)} candidate rows')
print(f'{data.subregion.nunique()} constituencies')
print(f'{data.winner.sum()} winners')
print()
print(data.loc[data.winner, 'party'].value_counts().to_string())

assert len(data) == 659
assert data.subregion.nunique() == 106
assert data.winner.sum() == 106

### 1. Plot the number of voters in each region

In [ ]:
by_region = data.groupby('region')['votes'].sum().sort_values(ascending=False)
by_region.plot(kind='bar', figsize=(12, 5), title='Votes cast per region');

assert by_region.sum() == data['votes'].sum()
assert by_region.index[0] == 'Budapest főváros'   # by far the biggest
print(by_region.head())

### 2. Counterfactual: who leads nationally without TISZA?

`~` negates a boolean filter. The answer is FIDESZ-KDNP - which is not a surprise, but the *interesting* number is the gap: with TISZA in, they are 1.1M votes behind; with TISZA out, they lead by nearly 1.9M.

In [ ]:
without_tisza = data.loc[~(data['party'] == 'TISZA')]

result = (without_tisza
          .groupby('party')['votes']
          .sum()
          .sort_values(ascending=False))

print(result.head())
print(f'\nwinner without TISZA: {result.index[0]}')

with_tisza = data.groupby('party')['votes'].sum().sort_values(ascending=False)
print(f'\nwith TISZA    : {with_tisza.index[0]} leads by '
      f'{with_tisza.iloc[0] - with_tisza.iloc[1]:,}')
print(f'without TISZA : {result.index[0]} leads by '
      f'{result.iloc[0] - result.iloc[1]:,}')

assert result.index[0] == 'FIDESZ-KDNP'
assert with_tisza.index[0] == 'TISZA'
assert 'TISZA' not in result.index

### 3. The same, region by region

`idxmax()` on a grouped sum gives the label of the biggest value - much neater than sorting and taking the head.

In [ ]:
by_region_party = (without_tisza
                   .groupby(['region', 'party'])['votes']
                   .sum())

leaders = by_region_party.groupby('region').idxmax().apply(lambda pair: pair[1])
print(leaders.to_string())
print()
print(leaders.value_counts().to_string())

assert len(leaders) == data.region.nunique()
# without TISZA, FIDESZ-KDNP would come first in every single region
assert set(leaders.unique()) == {'FIDESZ-KDNP'}

### 4. The most successful candidates (top 10)

In [ ]:
top10 = data.nlargest(10, 'votes')[['name', 'party', 'region', 'votes', 'winner']]
print(top10.to_string(index=False))

assert len(top10) == 10
assert top10['votes'].is_monotonic_decreasing
# raw vote counts favour the big city districts - keep this in mind for ex. 9
assert top10['winner'].all()

### 5. How many constituencies did each party contest?

`nunique()` rather than `count()`: `count()` would count rows, and a party can only field one candidate per constituency anyway - but `nunique()` says what we mean.

In [ ]:
contested = (data
             .groupby('party')['subregion']
             .nunique()
             .sort_values(ascending=False))

print(contested.to_string())

assert contested.max() <= 106
# the two big parties ran everywhere
assert contested['TISZA'] == 106 and contested['FIDESZ-KDNP'] == 106

### 6. The closest races

`groupby(...).head(2)` keeps the first two rows *of each group* - so after sorting by votes it gives the winner and the runner-up of every constituency.

The tightest race in the country was decided by **211 votes**.

In [ ]:
ranked = data.sort_values('votes', ascending=False)
top_two = ranked.groupby('subregion').head(2)

margins = (top_two
           .groupby('subregion')['votes']
           .agg(lambda votes: votes.iloc[0] - votes.iloc[1])
           .sort_values())

print(margins.head(7).to_string())
print(f'\ndecided by fewer than 1000 votes: {(margins < 1000).sum()} constituencies')
print(f'decided by fewer than  500 votes: {(margins < 500).sum()} constituencies')

assert len(margins) == 106
assert margins.min() == 211
assert (margins < 1000).sum() == 7

Who won those knife-edge seats?

In [ ]:
closest = margins.head(7).index
winners_of_close = data.loc[data.winner & data.subregion.isin(closest),
                            ['subregion', 'name', 'party', 'votes']]
print(winners_of_close.to_string(index=False))
print()
print(winners_of_close['party'].value_counts().to_string())

assert len(winners_of_close) == 7

### 7. The best region for each party

The four steps in the notebook's hint, in order: group, sort, reset the index, then take the first row of each party.

In [ ]:
best_regions = (data
                .groupby(['party', 'region'])['votes']
                .sum()
                .sort_values(ascending=False)
                .reset_index()
                .groupby('party')
                .first())

print(best_regions.to_string())

assert len(best_regions) == data.party.nunique()
assert best_regions.loc['TISZA', 'region'] == 'Budapest főváros'

### 8. The strongest independent candidate

Hadházy Ákos, Budapest 6 - **10 207 votes**. The next best independent managed 2 132, so he is roughly a **five times** outlier. He still did not win the seat.

In [ ]:
independents = (data.loc[data['party'] == 'Független jelölt']
                .sort_values('votes', ascending=False))

print(independents.head(5).to_string(index=False))

best = independents.iloc[0]
runner_up = independents.iloc[1]
print(f'\n{best["name"]}: {best["votes"]:,} votes')
print(f'next best independent: {runner_up["votes"]:,} '
      f'({best["votes"] / runner_up["votes"]:.1f}x smaller)')
print(f'did he win? {best["winner"]}')

assert best['name'] == 'HADHÁZY ÁKOS'
assert best['votes'] == 10207
assert not best['winner']
assert best['votes'] > 4 * runner_up['votes']

### 9. Votes are not the whole story: the vote *share*

`transform('sum')` is the key idea: unlike `sum()`, it gives back a value **for every row** of the group, so it lines up with the original dataframe and can go straight into a new column.

The top 5 by votes and the top 5 by share are **different people** - which is the whole point of the exercise.

In [ ]:
data['district_total'] = data.groupby('subregion')['votes'].transform('sum')
data['share'] = data['votes'] / data['district_total'] * 100

by_votes = data.nlargest(5, 'votes')[['name', 'party', 'votes', 'share']]
by_share = data.nlargest(5, 'share')[['name', 'party', 'votes', 'share']]

print('top 5 by votes:'); print(by_votes.round(1).to_string(index=False))
print('\ntop 5 by share:'); print(by_share.round(1).to_string(index=False))

# every district's shares must add up to 100%
totals = data.groupby('subregion')['share'].sum()
assert ((totals - 100).abs() < 1e-9).all()
# the two rankings are genuinely different
assert list(by_votes['name']) != list(by_share['name'])

hadhazy = data.loc[data['name'] == 'HADHÁZY ÁKOS'].iloc[0]
print(f"\nHadházy: {hadhazy['votes']:,} votes = {hadhazy['share']:.1f}% of his district")
assert round(hadhazy['share'], 1) == 16.1

### 10. Who overperformed their own party?

`transform('mean')` again, this time grouped by party. And the answer at the top of the list is **Hadházy** - 16.1% against an independent-candidate average of 0.9%, so **+15.2 points**. The same person the raw vote count in exercise 4 did not even put in the top 10.

In [ ]:
data['party_mean_share'] = data.groupby('party')['share'].transform('mean')
data['overperformance'] = data['share'] - data['party_mean_share']

top_overperformers = data.nlargest(5, 'overperformance')[
    ['name', 'party', 'share', 'party_mean_share', 'overperformance']]
print(top_overperformers.round(1).to_string(index=False))

best = top_overperformers.iloc[0]
assert best['name'] == 'HADHÁZY ÁKOS'
assert round(best['overperformance'], 1) == 15.2
# he was nowhere near the top on raw votes
assert 'HADHÁZY ÁKOS' not in set(data.nlargest(10, 'votes')['name'])
print('\nSame candidate as exercise 8 - and invisible in exercise 4.')

### 11. The best losers

Nagy István (FIDESZ-KDNP) got **26 435** votes and still lost. That is more than twice Hadházy's total - but only 40.7% of his district against Hadházy's 16.1%.

Good question to leave hanging: which of the two performed better? There is no single right answer, which is exactly why you have to say which number you are quoting.

In [ ]:
losers = (data.loc[~data.winner]
          .nlargest(5, 'votes')[['name', 'party', 'subregion', 'votes', 'share']])
print(losers.round(1).to_string(index=False))

best_loser = losers.iloc[0]
print(f"\n{best_loser['name']}: {best_loser['votes']:,} votes "
      f"({best_loser['share']:.1f}%) - and lost")
print(f"Hadházy      : {hadhazy['votes']:,} votes "
      f"({hadhazy['share']:.1f}%) - and lost")

assert best_loser['votes'] == 26435
assert not data.loc[~data.winner, 'winner'].any()
assert best_loser['votes'] > 2 * hadhazy['votes']
assert best_loser['share'] > hadhazy['share']

### 12. Two data-quality traps

**a)** 20 rows have exactly 0 votes, spread over 18 constituencies. These are candidates who were registered but withdrew before the ballots were printed - the register still lists them, the count gives them nothing.

They matter: they drag every `mean()` down. Dropping them moves the independent average from 0.9% to 1.4%, and Hadházy's overperformance with it.

**b)** 7 names appear more than once - `NAGY FERENC` three times. One of them is a name shared with a nationally known politician in a completely different constituency. **Never join two datasets on people's names.**

In [ ]:
zeros = data.loc[data['votes'] == 0]
print(f'{len(zeros)} rows with 0 votes, in {zeros.subregion.nunique()} constituencies')
print(zeros['party'].value_counts().to_string())

# what dropping them does to exercise 10
independents_all = data.loc[data['party'] == 'Független jelölt', 'share'].mean()
independents_nonzero = data.loc[(data['party'] == 'Független jelölt')
                                & (data['votes'] > 0), 'share'].mean()
print(f'\nmean independent share  including zeros: {independents_all:.2f}%')
print(f'mean independent share  excluding zeros: {independents_nonzero:.2f}%')

assert len(zeros) == 20
assert zeros.subregion.nunique() == 18
assert independents_nonzero > independents_all

In [ ]:
duplicate_names = data['name'].value_counts()
duplicate_names = duplicate_names[duplicate_names > 1]

print(f'{len(duplicate_names)} names appear more than once:')
print(duplicate_names.to_string())

print('\nfor example:')
example = duplicate_names.index[duplicate_names.index.get_indexer(['MAGYAR PÉTER'])[0]] \
    if 'MAGYAR PÉTER' in duplicate_names.index else duplicate_names.index[0]
print(data.loc[data['name'] == example,
               ['name', 'party', 'subregion', 'votes', 'winner']].to_string(index=False))

assert len(duplicate_names) == 7
assert duplicate_names.max() == 3        # NAGY FERENC
# the same name, two different people, two different parties
shared = data.loc[data['name'] == example]
assert shared['party'].nunique() > 1 or shared['subregion'].nunique() > 1